# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/semana6')
print(os.listdir('.'))

Mounted at /content/drive
['data', 'artifacts', 'reports']


In [2]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [3]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [4]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [7]:
feature_columns = contract["features"]

# Árboles: solo imputar (no necesitan escalar)
tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), feature_columns)
])

# LogReg y KNN: imputar + escalar (son sensibles a la escala)
scaled_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns)
])

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si el scaler se ajusta antes con todo el train, calcula la media y
la desviación usando también los datos de validación de cada fold. Eso es *data leakage*:
el modelo "ve" información de la validación y el resultado sale optimista. Dentro del
pipeline, el scaler se ajusta solo con la parte de entrenamiento de cada fold.

## 3. Modelos candidatos

In [8]:
models = {
    "decision_tree": Pipeline([
        ("preprocess", tree_preprocessor),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocess", scaled_preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocess", scaled_preprocessor),
        ("model", KNeighborsClassifier()),
    ]),
}

## 4. Comparación inicial con validación cruzada

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv,
                            scoring="accuracy", return_train_score=True)
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "val_accuracy": scores["test_score"].mean(),
        "val_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).round(4)
print(initial_comparison)

                 model  train_accuracy  val_accuracy  val_std
0        decision_tree          1.0000        0.9022   0.0586
1  logistic_regression          1.0000        0.9791   0.0277
2                  knn          0.9771        0.9507   0.0353


## 5. Optimización de hiperparámetros

In [10]:
param_grids = {
    "decision_tree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 3, 5],
    },
    "logistic_regression": {"model__C": [0.01, 0.1, 1, 10]},
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

searches, rows = {}, []
for name, pipeline in models.items():
    search = GridSearchCV(pipeline, param_grids[name], cv=cv,
                          scoring="accuracy", return_train_score=True)
    search.fit(X_train, y_train)
    searches[name] = search
    best = search.best_index_
    rows.append({
        "model": name,
        "best_params": search.best_params_,
        "train_accuracy": search.cv_results_["mean_train_score"][best],
        "val_accuracy": search.best_score_,
        "val_std": search.cv_results_["std_test_score"][best],
    })

tuned_comparison = pd.DataFrame(rows).round(4)
print(tuned_comparison.to_string())

                 model                                             best_params  train_accuracy  val_accuracy  val_std
0        decision_tree   {'model__max_depth': 4, 'model__min_samples_leaf': 1}          0.9982        0.9022   0.0586
1  logistic_regression                                         {'model__C': 1}          1.0000        0.9791   0.0277
2                  knn  {'model__n_neighbors': 7, 'model__weights': 'uniform'}          0.9648        0.9579   0.0405


## 6. Selección y persistencia del campeón

In [11]:
champion_name = max(searches, key=lambda n: searches[n].best_score_)
champion = searches[champion_name].best_estimator_
print("Campeón:", champion_name)

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy": float(searches[champion_name].best_score_),
    "cv_folds": 5,
    "random_state": RANDOM_STATE,
    "features": feature_columns,
}
with open(ARTIFACTS_DIR / "training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

Campeón: logistic_regression


In [ ]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

**Justificación de la selección:**

El campeón es regresión logística (C = 1), con 97.9 % de accuracy en CV.
- **Desempeño:** el mejor de los tres (KNN 95.8 %, árbol 90.2 %).
- **Variabilidad:** la desviación más baja (±2.8 %), así que es el más estable entre folds.
- **Sobreajuste:** train 100 % vs validación 97.9 %, una brecha pequeña. El árbol tiene
  una brecha mayor (99.8 % vs 90.2 %).
- **Complejidad e interpretabilidad:** es un modelo simple y sus coeficientes muestran
  qué variables pesan más en cada clase.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.